# Three-Class Username NER Workflow

**Project:** *Alias Construction as Forensic Evidence: A Computational Corpus -Based Study on Forensic Onomastics*

This notebook runs the same named-entity recognition (NER) procedure on:

- **Class A:** documented criminal/offender usernames and aliases
- **Class B:** public-figure handles from X and YouTube
- **Class C:** a fixed-seed probability sample of the Kaggle Reddit username corpus

It compares **raw handles** with **lightly normalized handles** (camel-case, delimiter, and letter–digit splitting). It produces aggregate detection rates, Wilson 95% confidence intervals, entity-label tables, pairwise Fisher tests, raw-versus-normalized McNemar tests, figures, and a private manual-validation template.

## Before running

1. Open this notebook in Google Colab.
2. Select **Runtime → Change runtime type → T4 GPU**.
3. Run the cells from top to bottom.
4. When prompted, upload the Class A and Class B CSV files.

**Ethics:** The Reddit workflow reads only the username column. It does not retain URLs, IDs, histories, posts, or timestamps; does not cross-link accounts; and saves aggregate results by default. Do not quote individual Reddit usernames in the thesis.

**Scope note:** “All three classes” means that every class is analyzed with the same procedure. Class A and B are analyzed exhaustively after exact-deduplication. Class C is sampled because transformer NER over all ~25.9 million Reddit usernames would be disproportionate for a Colab workflow.


In [ ]:
%pip -q install -U "spacy[transformers]>=3.7,<4" kagglehub scipy seaborn tqdm scikit-learn
!python -m spacy download en_core_web_trf


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 4.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.4/313.4 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 36.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is th

In [ ]:
from pathlib import Path

SEED = 42
MODEL_NAME = "en_core_web_trf"

CLASS_C_SAMPLE_MODE = "match_class_b"
CLASS_C_FIXED_N = 20_000
REDDIT_READ_CHUNK_SIZE = 500_000

NER_BATCH_SIZE = 64
MANUAL_VALIDATION_N_PER_CLASS = 100

SAVE_ROW_LEVEL_OUTPUTS = False

OUTPUT_DIR = Path("/content/ner_analysis_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")


Configuration loaded.


## 1. Upload and locate Class A and Class B




In [ ]:
import pandas as pd

try:
    from google.colab import files
    print("Select the Class A and Class B CSV files.")
    uploaded = files.upload()
except ImportError:
    print("Not running in Colab; searching the current directory.")

def inspect_csv_columns(path):
    try:
        return list(pd.read_csv(path, nrows=0).columns)
    except Exception:
        return []

def locate_input_files():
    search_roots = [Path("/content"), Path.cwd()]
    candidates = []
    for root in search_roots:
        if root.exists():
            candidates.extend(root.glob("*.csv"))

    candidates = list(dict.fromkeys(p.resolve() for p in candidates))
    class_a_file = None
    class_b_file = None

    for path in candidates:
        columns = inspect_csv_columns(path)
        if "Username" in columns and class_a_file is None:
            class_a_file = path
        if {"handle", "platform"}.issubset(columns) and class_b_file is None:
            class_b_file = path

    if class_a_file is None or class_b_file is None:
        raise FileNotFoundError(
            "Could not identify both inputs. Class A needs a 'Username' "
            "column; Class B needs 'handle' and 'platform' columns."
        )
    return class_a_file, class_b_file

CLASS_A_FILE, CLASS_B_FILE = locate_input_files()
print("Class A file:", CLASS_A_FILE.name)
print("Class B file:", CLASS_B_FILE.name)


Select the Class A and Class B CSV files.


Saving Class_A_Criminal_Usernames_Master_Database.csv to Class_A_Criminal_Usernames_Master_Database (1).csv


FileNotFoundError: Could not identify both inputs. Class A needs a 'Username' column; Class B needs 'handle' and 'platform' columns.

In [ ]:

class_a_raw = pd.read_csv(
    CLASS_A_FILE,
    usecols=["Username"],
    dtype={"Username": "string"},
    low_memory=False,
)
class_b_raw = pd.read_csv(
    CLASS_B_FILE,
    usecols=["handle", "platform"],
    dtype={"handle": "string", "platform": "string"},
    low_memory=False,
)

def clean_text_column(series):
    cleaned = series.astype("string").str.strip()
    return cleaned[cleaned.notna() & cleaned.ne("")]

class_a = pd.DataFrame({"username": clean_text_column(class_a_raw["Username"])})
class_a["class"] = "A"
class_a["platform"] = "not_applicable"
class_a = class_a.drop_duplicates(["username"]).reset_index(drop=True)

class_b = class_b_raw.rename(columns={"handle": "username"}).copy()
class_b["username"] = class_b["username"].astype("string").str.strip()
class_b["platform"] = class_b["platform"].astype("string").str.strip()
class_b = class_b[
    class_b["username"].notna() & class_b["username"].ne("")
].copy()
class_b["class"] = "B"
class_b = class_b.drop_duplicates(["platform", "username"]).reset_index(drop=True)

input_audit = pd.DataFrame(
    [
        {
            "class": "A",
            "input_rows": len(class_a_raw),
            "retained_records": len(class_a),
            "rule": "unique username",
        },
        {
            "class": "B",
            "input_rows": len(class_b_raw),
            "retained_records": len(class_b),
            "rule": "unique platform + username",
        },
    ]
)

print(input_audit.to_string(index=False))
print("\nClass B retained records by platform:")
print(class_b["platform"].value_counts().sort_index().to_string())


## 2. Download and sample Class C

The sampler streams through the entire Reddit file and assigns independent random priorities with seed 42. It retains the lowest priorities, producing a reproducible probability sample without loading the 376 MB file into memory.


In [ ]:
import kagglehub

DATA_DIR = Path(
    kagglehub.dataset_download("colinmorris/reddit-usernames")
)

csv_files = list(DATA_DIR.rglob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No CSV file was found under {DATA_DIR}")

known = [p for p in csv_files if p.name.lower() == "users.csv"]
REDDIT_FILE = known[0] if known else max(csv_files, key=lambda p: p.stat().st_size)

print("Reddit source file:", REDDIT_FILE.name)
print("File size (MB):", round(REDDIT_FILE.stat().st_size / 1_000_000, 2))


In [ ]:
import numpy as np

def detect_username_column(csv_path):
    columns = list(pd.read_csv(csv_path, nrows=0).columns)
    preferred = ["username", "user_name", "user", "handle", "name"]
    lower_map = {str(col).lower().strip(): col for col in columns}
    for candidate in preferred:
        if candidate in lower_map:
            return lower_map[candidate]
    if len(columns) == 1:
        return columns[0]
    raise ValueError(
        f"Could not identify the username column. Found: {columns}"
    )

def priority_sample_csv(csv_path, n, seed=42, chunksize=500_000):
    username_col = detect_username_column(csv_path)
    rng = np.random.default_rng(seed)

    keep_n = max(n, int(np.ceil(n * 1.10)))
    best_values = np.empty(0, dtype=object)
    best_keys = np.empty(0, dtype=np.float64)
    processed_valid = 0

    reader = pd.read_csv(
        csv_path,
        usecols=[username_col],
        dtype={username_col: "string"},
        chunksize=chunksize,
        low_memory=False,
        on_bad_lines="skip",
    )

    for chunk_number, chunk in enumerate(reader, start=1):
        values = chunk[username_col].astype("string").str.strip()
        values = values[
            values.notna()
            & values.ne("")
            & values.str.len().between(1, 100)
        ]
        array = values.to_numpy(dtype=object)
        keys = rng.random(len(array))
        processed_valid += len(array)

        if len(array) > keep_n:
            idx = np.argpartition(keys, keep_n - 1)[:keep_n]
            array = array[idx]
            keys = keys[idx]

        combined_values = np.concatenate([best_values, array])
        combined_keys = np.concatenate([best_keys, keys])

        if len(combined_keys) > keep_n:
            idx = np.argpartition(combined_keys, keep_n - 1)[:keep_n]
            best_values = combined_values[idx]
            best_keys = combined_keys[idx]
        else:
            best_values = combined_values
            best_keys = combined_keys

        if chunk_number % 5 == 0:
            print(
                f"Processed {processed_valid:,} valid username rows; "
                f"retained {len(best_values):,} candidates."
            )

    sampled = pd.DataFrame(
        {"username": best_values, "_priority": best_keys}
    ).sort_values("_priority")
    sampled = sampled.drop_duplicates("username").head(n).drop(columns="_priority")

    if len(sampled) < n:
        print(
            f"Warning: requested {n:,} unique strings but retained "
            f"{len(sampled):,} after cleaning and deduplication."
        )
    return sampled.reset_index(drop=True), username_col, processed_valid

reddit_sample_n = (
    len(class_b)
    if CLASS_C_SAMPLE_MODE == "match_class_b"
    else CLASS_C_FIXED_N
)

class_c, reddit_username_column, reddit_valid_rows_scanned = priority_sample_csv(
    REDDIT_FILE,
    n=reddit_sample_n,
    seed=SEED,
    chunksize=REDDIT_READ_CHUNK_SIZE,
)
class_c["class"] = "C"
class_c["platform"] = "reddit"

input_audit = pd.concat(
    [
        input_audit,
        pd.DataFrame(
            [
                {
                    "class": "C",
                    "input_rows": reddit_valid_rows_scanned,
                    "retained_records": len(class_c),
                    "rule": f"fixed-seed priority sample (seed={SEED})",
                }
            ]
        ),
    ],
    ignore_index=True,
)

print("\nFinal analysis counts:")
print(input_audit.to_string(index=False))


## 3. Prepare raw and normalized NER inputs



In [ ]:
import hashlib
import re

def normalize_username(value):
    text = str(value).strip()
    text = re.sub(r"^@", "", text)
    text = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", text)
    text = re.sub(r"[_\-.]+", " ", text)
    text = re.sub(
        r"(?<=[^\W\d_])(?=\d)|(?<=\d)(?=[^\W\d_])",
        " ",
        text,
        flags=re.UNICODE,
    )
    text = re.sub(r"[^\w\s']+", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text

corpus = pd.concat(
    [
        class_a[["class", "platform", "username"]],
        class_b[["class", "platform", "username"]],
        class_c[["class", "platform", "username"]],
    ],
    ignore_index=True,
)
corpus["normalized_username"] = corpus["username"].map(normalize_username)

def make_record_id(row):
    payload = "\x1f".join(
        [str(row["class"]), str(row["platform"]), str(row["username"])]
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:20]

corpus["record_id"] = corpus.apply(make_record_id, axis=1)

raw_inputs = corpus.assign(variant="raw", ner_text=corpus["username"])
normalized_inputs = corpus.assign(
    variant="normalized",
    ner_text=corpus["normalized_username"],
)
ner_inputs = pd.concat([raw_inputs, normalized_inputs], ignore_index=True)
ner_inputs["ner_text"] = ner_inputs["ner_text"].fillna("").astype(str)

print("Unique source records by class:")
print(corpus["class"].value_counts().sort_index().to_string())
print("\nNER texts including both variants:", f"{len(ner_inputs):,}")


## 4. Load the NER model

The default is spaCy's English transformer model. Only its transformer and NER components are retained. The model is English-oriented, so outputs for non-English forms must be interpreted cautiously.


In [ ]:
import spacy

gpu_active = spacy.prefer_gpu()
if not gpu_active:
    print(
        "WARNING: No GPU was detected. The transformer model will be slow. "
        "In Colab, select Runtime > Change runtime type > T4 GPU, then rerun."
    )

model_pipeline = spacy.info(MODEL_NAME).get("pipeline", [])
keep_components = {"transformer", "tok2vec", "ner"}
excluded_components = [
    component for component in model_pipeline
    if component not in keep_components
]

nlp = spacy.load(MODEL_NAME, exclude=excluded_components)
print("Model:", MODEL_NAME)
print("Active pipeline:", nlp.pipe_names)
print("GPU active:", gpu_active)


## 5. Run NER identically across the three classes

The primary analytical set includes `PERSON`, `ORG`, `GPE`, `LOC`, `FAC`, `NORP`, `PRODUCT`, `EVENT`, `WORK_OF_ART`, `LAW`, `LANGUAGE`, and `DATE`. Purely numeric labels such as `CARDINAL`, `ORDINAL`, `MONEY`, `PERCENT`, `QUANTITY`, and `TIME` remain available in the all-label output but are excluded from the primary “named-entity cue” rate.


In [ ]:
from tqdm.auto import tqdm

ANALYTIC_LABELS = {
    "PERSON",
    "ORG",
    "GPE",
    "LOC",
    "FAC",
    "NORP",
    "PRODUCT",
    "EVENT",
    "WORK_OF_ART",
    "LAW",
    "LANGUAGE",
    "DATE",
}

def run_ner(texts):
    records = []
    docs = nlp.pipe(
        texts,
        batch_size=NER_BATCH_SIZE,
        n_process=1,
    )
    for doc in tqdm(docs, total=len(texts), desc="Running NER"):
        all_labels = [ent.label_ for ent in doc.ents]
        analytic_labels = [
            ent.label_ for ent in doc.ents
            if ent.label_ in ANALYTIC_LABELS
        ]
        records.append(
            {
                "all_entity_labels": all_labels,
                "analytic_entity_labels": analytic_labels,
                "all_entity_count": len(all_labels),
                "analytic_entity_count": len(analytic_labels),
                "has_any_model_entity": int(bool(all_labels)),
                "has_analytic_entity": int(bool(analytic_labels)),
                "primary_analytic_label": (
                    analytic_labels[0] if analytic_labels else "NONE"
                ),
            }
        )
    return pd.DataFrame(records)

ner_predictions = run_ner(ner_inputs["ner_text"].tolist())
ner_results = pd.concat(
    [ner_inputs.reset_index(drop=True), ner_predictions],
    axis=1,
)

print("NER complete.")
print(
    ner_results.groupby(["class", "variant"])["has_analytic_entity"]
    .agg(["count", "sum"])
    .to_string()
)


## 6. Aggregate results and statistical comparisons

- Wilson intervals quantify uncertainty around each entity-detection rate.
- Fisher's exact tests compare detection rates between classes.
- Holm adjustment controls family-wise error across the six pairwise tests.
- McNemar tests assess whether light normalization changes detection for the same usernames.
- Effect sizes are reported alongside p-values.


In [ ]:
import math
from itertools import combinations

from scipy.stats import binomtest, fisher_exact, norm

def wilson_interval(successes, n, alpha=0.05):
    if n == 0:
        return np.nan, np.nan
    z = norm.ppf(1 - alpha / 2)
    p = successes / n
    denominator = 1 + (z ** 2 / n)
    center = (p + z ** 2 / (2 * n)) / denominator
    margin = (
        z
        * math.sqrt(
            (p * (1 - p) / n) + (z ** 2 / (4 * n ** 2))
        )
        / denominator
    )
    return max(0.0, center - margin), min(1.0, center + margin)

def holm_adjust(p_values):
    values = np.asarray(p_values, dtype=float)
    order = np.argsort(values)
    ranked = values[order]
    m = len(values)
    adjusted_ranked = np.maximum.accumulate(
        [(m - index) * p for index, p in enumerate(ranked)]
    )
    adjusted_ranked = np.minimum(adjusted_ranked, 1.0)
    adjusted = np.empty(m, dtype=float)
    adjusted[order] = adjusted_ranked
    return adjusted

def detection_summary(frame, group_columns):
    rows = []
    for keys, group in frame.groupby(group_columns, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        n = len(group)
        successes = int(group["has_analytic_entity"].sum())
        low, high = wilson_interval(successes, n, alpha=0.05)
        row = dict(zip(group_columns, keys))
        row.update(
            {
                "n_usernames": n,
                "usernames_with_entity": successes,
                "entity_rate": successes / n if n else np.nan,
                "ci95_low": float(low),
                "ci95_high": float(high),
                "mean_analytic_entities": float(
                    group["analytic_entity_count"].mean()
                ),
            }
        )
        rows.append(row)
    return pd.DataFrame(rows)

def label_summary(frame, group_columns):
    exploded = frame[
        frame["analytic_entity_count"].gt(0)
    ].explode("analytic_entity_labels")
    if exploded.empty:
        return pd.DataFrame(
            columns=group_columns
            + [
                "entity_label",
                "n_entities",
                "usernames_with_label",
                "pct_of_analytic_entities",
                "pct_of_usernames",
            ]
        )

    grouped = (
        exploded.groupby(
            group_columns + ["analytic_entity_labels"],
            dropna=False,
        )
        .agg(
            n_entities=("analytic_entity_labels", "size"),
            usernames_with_label=("record_id", "nunique"),
        )
        .reset_index()
        .rename(columns={"analytic_entity_labels": "entity_label"})
    )
    denominators = (
        frame.groupby(group_columns, dropna=False)
        .size()
        .rename("n_usernames")
        .reset_index()
    )
    entity_totals = (
        grouped.groupby(group_columns, dropna=False)["n_entities"]
        .sum()
        .rename("total_analytic_entities")
        .reset_index()
    )
    grouped = grouped.merge(denominators, on=group_columns)
    grouped = grouped.merge(entity_totals, on=group_columns)
    grouped["pct_of_analytic_entities"] = (
        grouped["n_entities"]
        / grouped["total_analytic_entities"]
        * 100
    )
    grouped["pct_of_usernames"] = (
        grouped["usernames_with_label"]
        / grouped["n_usernames"]
        * 100
    )
    return grouped

class_detection_summary = detection_summary(
    ner_results, ["class", "variant"]
)
class_label_summary = label_summary(
    ner_results, ["class", "variant"]
)

class_b_platform_detection_summary = detection_summary(
    ner_results[ner_results["class"].eq("B")],
    ["platform", "variant"],
)
class_b_platform_label_summary = label_summary(
    ner_results[ner_results["class"].eq("B")],
    ["platform", "variant"],
)

pairwise_rows = []
for variant in ["raw", "normalized"]:
    subset = ner_results[ner_results["variant"].eq(variant)]
    for class_1, class_2 in combinations(["A", "B", "C"], 2):
        g1 = subset[subset["class"].eq(class_1)]
        g2 = subset[subset["class"].eq(class_2)]
        s1 = int(g1["has_analytic_entity"].sum())
        s2 = int(g2["has_analytic_entity"].sum())
        n1, n2 = len(g1), len(g2)
        table = [[s1, n1 - s1], [s2, n2 - s2]]
        odds_ratio, p_value = fisher_exact(table, alternative="two-sided")
        p1 = s1 / n1 if n1 else np.nan
        p2 = s2 / n2 if n2 else np.nan
        cohen_h = 2 * (
            math.asin(math.sqrt(p1))
            - math.asin(math.sqrt(p2))
        )
        pairwise_rows.append(
            {
                "variant": variant,
                "class_1": class_1,
                "class_2": class_2,
                "n_1": n1,
                "n_2": n2,
                "rate_1": p1,
                "rate_2": p2,
                "rate_difference_1_minus_2": p1 - p2,
                "odds_ratio": odds_ratio,
                "cohen_h": cohen_h,
                "p_value_fisher": p_value,
            }
        )

pairwise_tests = pd.DataFrame(pairwise_rows)
pairwise_tests["p_value_holm"] = holm_adjust(
    pairwise_tests["p_value_fisher"]
)

mcnemar_rows = []
for class_name in ["A", "B", "C"]:
    wide = (
        ner_results[ner_results["class"].eq(class_name)]
        .pivot(
            index="record_id",
            columns="variant",
            values="has_analytic_entity",
        )
        .dropna()
        .astype(int)
    )
    table = np.array(
        [
            [
                int(((wide["raw"] == 0) & (wide["normalized"] == 0)).sum()),
                int(((wide["raw"] == 0) & (wide["normalized"] == 1)).sum()),
            ],
            [
                int(((wide["raw"] == 1) & (wide["normalized"] == 0)).sum()),
                int(((wide["raw"] == 1) & (wide["normalized"] == 1)).sum()),
            ],
        ]
    )
    discordant = int(table[0, 1] + table[1, 0])
    if discordant:
        test = binomtest(
            min(int(table[0, 1]), int(table[1, 0])),
            n=discordant,
            p=0.5,
            alternative="two-sided",
        )
        statistic = min(int(table[0, 1]), int(table[1, 0]))
        p_value = float(test.pvalue)
    else:
        statistic = 0
        p_value = 1.0
    mcnemar_rows.append(
        {
            "class": class_name,
            "n_paired": len(wide),
            "raw_no_normalized_yes": int(table[0, 1]),
            "raw_yes_normalized_no": int(table[1, 0]),
            "statistic": statistic,
            "p_value": p_value,
        }
    )

raw_vs_normalized_tests = pd.DataFrame(mcnemar_rows)
raw_vs_normalized_tests["p_value_holm"] = holm_adjust(
    raw_vs_normalized_tests["p_value"]
)

display(
    class_detection_summary.assign(
        entity_rate_pct=lambda d: (d["entity_rate"] * 100).round(2),
        ci95_low_pct=lambda d: (d["ci95_low"] * 100).round(2),
        ci95_high_pct=lambda d: (d["ci95_high"] * 100).round(2),
    )[
        [
            "class",
            "variant",
            "n_usernames",
            "usernames_with_entity",
            "entity_rate_pct",
            "ci95_low_pct",
            "ci95_high_pct",
        ]
    ]
)
display(pairwise_tests.round(5))
display(raw_vs_normalized_tests.round(5))


## 7. Create thesis-ready aggregate figures

Percentages represent the share of usernames for which the model detected at least one label from the predefined analytical set. They are model outputs, not verified identity disclosures.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")
class_order = ["A", "B", "C"]
variant_order = ["raw", "normalized"]
colors = {"raw": "#4C78A8", "normalized": "#F58518"}

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(class_order))
width = 0.36

for index, variant in enumerate(variant_order):
    data = (
        class_detection_summary[
            class_detection_summary["variant"].eq(variant)
        ]
        .set_index("class")
        .reindex(class_order)
    )
    positions = x + (index - 0.5) * width
    values = data["entity_rate"].to_numpy() * 100
    lower = (data["entity_rate"] - data["ci95_low"]).to_numpy() * 100
    upper = (data["ci95_high"] - data["entity_rate"]).to_numpy() * 100
    ax.bar(
        positions,
        values,
        width,
        label=variant.title(),
        color=colors[variant],
        yerr=np.vstack([lower, upper]),
        capsize=4,
    )

ax.set_xticks(x, ["Class A", "Class B", "Class C"])
ax.set_ylabel("Usernames with an NER cue (%)")
ax.set_xlabel("")
ax.set_title("NER Detection Rate by Username Class")
ax.legend(title="Input")
fig.tight_layout()
detection_plot_path = OUTPUT_DIR / "ner_detection_rates.png"
fig.savefig(detection_plot_path, dpi=300, bbox_inches="tight")
plt.show()

normalized_labels = class_label_summary[
    class_label_summary["variant"].eq("normalized")
].copy()
top_labels = (
    normalized_labels.groupby("entity_label")["n_entities"]
    .sum()
    .nlargest(10)
    .index
)
heatmap_data = (
    normalized_labels[
        normalized_labels["entity_label"].isin(top_labels)
    ]
    .pivot(
        index="entity_label",
        columns="class",
        values="pct_of_usernames",
    )
    .reindex(columns=class_order)
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(9, max(4.5, len(heatmap_data) * 0.55)))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    cbar_kws={"label": "% of usernames"},
    ax=ax,
)
ax.set_title("Normalized-Input NER Labels by Class")
ax.set_xlabel("Username class")
ax.set_ylabel("NER label")
fig.tight_layout()
label_plot_path = OUTPUT_DIR / "ner_normalized_label_heatmap.png"
fig.savefig(label_plot_path, dpi=300, bbox_inches="tight")
plt.show()


## 8. Save outputs and build the private validation sample

The ZIP contains aggregate tables, tests, figures, configuration metadata, a methods paragraph, and a 100-per-class validation template. The validation template contains username strings because human checking requires seeing the analyzed form.

In [ ]:
import json
import shutil

input_audit.to_csv(OUTPUT_DIR / "input_audit.csv", index=False)
class_detection_summary.to_csv(
    OUTPUT_DIR / "ner_class_detection_summary.csv", index=False
)
class_label_summary.to_csv(
    OUTPUT_DIR / "ner_class_label_summary.csv", index=False
)
class_b_platform_detection_summary.to_csv(
    OUTPUT_DIR / "ner_class_b_platform_detection_summary.csv",
    index=False,
)
class_b_platform_label_summary.to_csv(
    OUTPUT_DIR / "ner_class_b_platform_label_summary.csv",
    index=False,
)
pairwise_tests.to_csv(
    OUTPUT_DIR / "ner_pairwise_fisher_tests.csv", index=False
)
raw_vs_normalized_tests.to_csv(
    OUTPUT_DIR / "ner_raw_vs_normalized_mcnemar_tests.csv",
    index=False,
)

raw_predictions = (
    ner_results[ner_results["variant"].eq("raw")]
    [
        [
            "record_id",
            "primary_analytic_label",
            "has_analytic_entity",
        ]
    ]
    .rename(
        columns={
            "primary_analytic_label": "raw_predicted_primary_label",
            "has_analytic_entity": "raw_model_has_entity",
        }
    )
)
normalized_predictions = (
    ner_results[ner_results["variant"].eq("normalized")]
    [
        [
            "record_id",
            "primary_analytic_label",
            "has_analytic_entity",
        ]
    ]
    .rename(
        columns={
            "primary_analytic_label": "normalized_predicted_primary_label",
            "has_analytic_entity": "normalized_model_has_entity",
        }
    )
)
validation_pool = (
    corpus.merge(raw_predictions, on="record_id")
    .merge(normalized_predictions, on="record_id")
)

validation_parts = []
for class_index, class_name in enumerate(["A", "B", "C"]):
    group = validation_pool[
        validation_pool["class"].eq(class_name)
    ]
    take_n = min(MANUAL_VALIDATION_N_PER_CLASS, len(group))
    validation_parts.append(
        group.sample(n=take_n, random_state=SEED + class_index)
    )

manual_validation = pd.concat(
    validation_parts, ignore_index=True
).sort_values(["class", "record_id"])
manual_validation["human_has_entity"] = ""
manual_validation["human_primary_label"] = ""
manual_validation["reviewer_notes"] = ""
manual_validation.to_csv(
    OUTPUT_DIR / "ner_manual_validation_private.csv",
    index=False,
)

if SAVE_ROW_LEVEL_OUTPUTS:
    private_detail = ner_results.copy()
    private_detail["all_entity_labels"] = private_detail[
        "all_entity_labels"
    ].map(lambda labels: "|".join(labels))
    private_detail["analytic_entity_labels"] = private_detail[
        "analytic_entity_labels"
    ].map(lambda labels: "|".join(labels))
    private_detail.to_csv(
        OUTPUT_DIR / "ner_row_level_private.csv",
        index=False,
    )

configuration = {
    "seed": SEED,
    "model_name": MODEL_NAME,
    "spacy_version": spacy.__version__,
    "gpu_active": bool(gpu_active),
    "class_c_sample_mode": CLASS_C_SAMPLE_MODE,
    "class_c_sample_n": int(len(class_c)),
    "reddit_rows_scanned": int(reddit_valid_rows_scanned),
    "reddit_username_column": str(reddit_username_column),
    "normalization": (
        "leading-at removal; camel-case, delimiter, and "
        "letter-digit splitting; punctuation cleanup; no title-casing"
    ),
    "analytic_labels": sorted(ANALYTIC_LABELS),
    "save_row_level_outputs": SAVE_ROW_LEVEL_OUTPUTS,
}
(OUTPUT_DIR / "ner_configuration.json").write_text(
    json.dumps(configuration, indent=2),
    encoding="utf-8",
)

counts = corpus["class"].value_counts().reindex(["A", "B", "C"])
methods_paragraph = (
    "Named-entity recognition (NER) was applied separately to three "
    "username classes using spaCy's English transformer model "
    f"({MODEL_NAME}): Class A (n={counts['A']:,}), Class B "
    f"(n={counts['B']:,}), and a fixed-seed probability sample of "
    f"Class C (n={counts['C']:,}; seed={SEED}). Class A and B were "
    "analyzed exhaustively after exact-deduplication, while Class C "
    "was sampled by streaming through the full Reddit username file "
    "and retaining records by random priority. Only username strings "
    "were read from the Reddit data. Each username was analyzed in "
    "raw form and after light normalization that split camel case, "
    "delimiters, and letter-digit boundaries without inventing "
    "capitalization. Results were summarized as aggregate label and "
    "detection rates with Wilson 95% confidence intervals. "
    "Between-class differences were evaluated using Fisher's exact "
    "tests with Holm-adjusted p-values, and raw-versus-normalized "
    "differences were evaluated using paired McNemar tests. NER "
    "predictions were treated as exploratory model-detected cues, "
    "not as verified identity attributes."
)
(OUTPUT_DIR / "ner_methods_paragraph.txt").write_text(
    methods_paragraph,
    encoding="utf-8",
)

zip_path = Path(
    shutil.make_archive(
        str(OUTPUT_DIR),
        "zip",
        root_dir=OUTPUT_DIR,
    )
)
print("Saved:", zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Download manually from:", zip_path)
